In [1]:
import sys
import os

sys.path.append("..")

In [2]:
import pickle
import pandas as pd
from tqdm import tqdm
import re
from awms.utils import extract_boxed, math_equal

from langchain_community.llms import VLLMOpenAI

In [3]:
job_id = "1497a141-3160-4896-8516-f7b7c3923265"
model_name = "llama_3_1_70b_instruct"
api_key = "8o30OElfDYV_D6YbbznT0A:GDC2BsXIfSdfjv9iWka3V4MkazpvHfe0cCwXohzbP0Q"

llm = VLLMOpenAI(
    openai_api_key=api_key,
    openai_api_base="https://" + job_id + "-8000.job.console.elementai.com/v1",
    model_name=model_name,
    temperature=0.1,
    max_tokens=1000,
    model_kwargs={},
)


def run_prompt(prompt):
    return llm.invoke(f"<|user|>\n{prompt}<|end|>\n<|assistant|>\n")

In [4]:
def convert_to_float(number, decimal_places):
    try:
        return round(float(number), decimal_places)
    except:
        return None


def gsm8k_true_response(text, decimal_places=3):
    pattern = r"####\s*([0-9]*\.?[0-9]+)"
    match = re.search(pattern, text)
    if match:
        number = match.group(1)
        return convert_to_float(number, decimal_places)
    else:
        return None

In [5]:
data = pickle.load(open("../explanations.pkl", "rb"))
print(len(data))
print(data[-1][0], data[-1][1])

1319
1317 To solve the problem, we first broke it down into two subproblems. The first subproblem was to find the number of chickens (C) and cows (c) given that there are 20 animals in total, i.e., C + c = 20. We solved this subproblem by isolating one of the variables, C, in terms of the other variable, c. This gave us an expression for C in terms of c, which we could then use to find different values for C by substituting different values for c.

However, this subproblem alone did not provide a unique solution for C and c. Therefore, we needed to consider the second subproblem, which was to find the number of chickens and cows given that they have a total of 70 legs, with each chicken having 2 legs and each cow having 4 legs, i.e., 2C + 4c = 70.

To solve the second subproblem, we used the expression for C in terms of c that we found in the first subproblem and substituted it into the second equation. This allowed us to solve for c, which we found to be 15. We then used this value of

In [6]:
indexes = [x[0] for x in data]

In [7]:
math = pd.read_json("../data/problems/gsm8k/test.jsonl", orient="records", lines=True)
math = math.loc[indexes].reset_index(drop=True)
true_results = [gsm8k_true_response(text.replace(",", "")) for text in math["answer"]]
math["true_result"] = true_results
math.head()

,question,answer,idx,true_result
0,A robe takes 2 bolts of blue fiber and half th...,It takes 2/2=<<2/2=1>>1 bolt of white fiber\nS...,1,3.0
1,James decides to run 3 sprints 3 times a week....,He sprints 3*3=<<3*3=9>>9 times\nSo he runs 9*...,3,540.0
2,Janet’s ducks lay 16 eggs per day. She eats th...,Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eg...,0,18.0
3,Josh decides to try flipping a house. He buys...,The cost of the house and repairs came out to ...,2,70000.0
4,"Every day, Wendi feeds each of her chickens th...","If each chicken eats 3 cups of feed per day, t...",4,20.0


In [8]:
acc = []
llm_results = []

for true_result, (i, di) in tqdm(zip(true_results, data)):
    if di is None:
        acc.append(None)
        llm_results.append(None)
        continue
    llm_result = extract_boxed(di)
    if llm_result is None:
        acc.append(None)
        llm_results.append(None)
    else:
        llm_result = llm_result.replace("$", "").replace(",", "").replace("%", "").strip()
        llm_results.append(llm_result)
        acc.append(math_equal(str(true_result), llm_result, timeout=False))

math["llm_result"] = llm_results
math["acc"] = acc

1319it [00:01, 939.22it/s] 


In [9]:
math.head(10)

,question,answer,idx,true_result,llm_result,acc
0,A robe takes 2 bolts of blue fiber and half th...,It takes 2/2=<<2/2=1>>1 bolt of white fiber\nS...,1,3.0,3,True
1,James decides to run 3 sprints 3 times a week....,He sprints 3*3=<<3*3=9>>9 times\nSo he runs 9*...,3,540.0,540,True
2,Janet’s ducks lay 16 eggs per day. She eats th...,Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eg...,0,18.0,18,True
3,Josh decides to try flipping a house. He buys...,The cost of the house and repairs came out to ...,2,70000.0,70000,True
4,"Every day, Wendi feeds each of her chickens th...","If each chicken eats 3 cups of feed per day, t...",4,20.0,20,True
5,Toulouse has twice as many sheep as Charleston...,"If Seattle has 20 sheep, Charleston has 4 * 20...",6,260.0,260,True
6,Eliza's rate per hour for the first 40 hours s...,Eliza is entitled to 45 -40 = <<45-40=5>>5 hou...,9,460.0,460,True
7,A new program had 60 downloads in the first mo...,The number of downloads of the program in the ...,10,366.0,366,True
8,Kylar went to the store to buy glasses for his...,The discount price of one glass is 60/100 * 5 ...,5,64.0,64,True
9,Carla is downloading a 200 GB file. Normally s...,First find how many gigabytes are in 40% of th...,7,160.0,160,True


In [10]:
math[~math["acc"].fillna(False)].tail(20)

,question,answer,idx,true_result,llm_result,acc
1076,Jenny goes to the florist to buy some flowers....,Jenny has $25 because 5 x 5 = <<5*5=25>>25\nSh...,1075,16.0,0,False
1090,One meatball sub sandwich contains 4 meatballs...,3 less than ten meatball sub sandwiches is 10-...,1068,24.0,None,None
1110,The temperature was 2 degrees Celsius. It drop...,Start = +<<2=2>>2 degrees\nDrop 8 degrees = 2 ...,1113,NaN,-3,False
1111,"Pierre, Paul, and Jacques bought 12 kg of appl...",Peter wants 12 * 1/4 = <<12*1/4=3>>3 kg of app...,1112,5.0,126/25,False
1113,Lauren is saving 20% of every paycheck. How ma...,Lauren plans to live in retirement with 40% of...,1105,40.0,122/25,False
1117,Ellen decided to play a prank on her friend. S...,First figure out how many unshaken sodas were ...,1094,25.0,3333/100,False
1121,"Amalia, Megan, and Dior divided the home chore...",Megan took 2 hours longer than Amalia to do he...,1119,18.0,19,False
1130,A toy manufacturer receives an order for 400 t...,"In the first group of workers, each produces 6...",1122,18.0,2,False
1160,Abraham owns 80 square meters of unused land. ...,Abraham sold 1/2 x 80= <<1/2*80=40>>40 square ...,1161,170.0,140,False
1166,Carla just gave birth to identical octuplets. ...,First find the number of babies wearing purple...,1159,50.0,100,False


In [11]:
math["acc"].mean(), math["acc"].fillna(False).mean()

(0.9334862385321101, 0.9257012888551933)

In [12]:
# inds_fail = math[math["acc"].isna()].index.tolist()
inds_fail = math[math["acc"] == False].index.tolist()

In [13]:
ind = -2
print(math.loc[inds_fail[ind]].reset_index(drop=True)[0])
print(math.loc[inds_fail[ind]].reset_index(drop=True)[1])

The girls are trying to raise money for a carnival. Kim raises $320 more than Alexandra, who raises $430, and Maryam raises $400 more than Sarah, who raises $300. How much money, in dollars, did they all raise in total?
Kim raises 320+430=<<320+430=750>>750 dollars.
Maryam raises 400+300=<<400+300=700>>700 dollars.
They raise 750+430+400+700=<<750+430+400+700=2280>>2280 dollars.
#### 2280


In [14]:
ind = -2
print(true_results[inds_fail[ind]])
print()
print(data[inds_fail[ind]][1])

2280.0

To solve this problem, we need to break it down into smaller subproblems and solve each one step by step.

The first subproblem is to find the amount of money raised by Kim. We are given that Kim raises $320 more than Alexandra, who raises $430. To find Kim's amount, we simply add $320 to Alexandra's amount. This gives us Kim's amount as $430 + $320 = $750.

The second subproblem is to find the amount of money raised by Maryam. We are given that Maryam raises $400 more than Sarah, who raises $300. To find Maryam's amount, we simply add $400 to Sarah's amount. This gives us Maryam's amount as $300 + $400 = $700.

Now that we have found the amounts raised by Kim and Maryam, we can add up the amounts raised by all four girls to find the total amount. The total amount is the sum of Alexandra's amount, Kim's amount, Sarah's amount, and Maryam's amount. This gives us a total amount of $430 + $750 + $300 + $700 = $2180.

Therefore, the girls raised a total of $2180 for the carnival.



In [15]:
# res = run_prompt(f"""
# Solve the following math problem and explain your reasoning. The answer shuld be in the last line and in a between $result$.
# {math.iloc[inds_fail[ind]]["question"]}
# """)
# print(res)

In [16]:
extract_boxed(data[inds_fail[ind]][1])

'$2180'

In [17]:
math_equal(str(true_results[inds_fail[ind]]), extract_boxed(data[inds_fail[ind]][1]), timeout=False)

False

In [18]:
qs = [
    (i, q)
    for i, q in enumerate(math.iloc[inds_fail]["question"])
    if "how long" in q.lower() or "how much time" in q.lower()
]
qs

[(15,
  'It takes 20 minutes for the oil to heat up to 300 degrees.  It then takes 40% longer for the oil to heat up to the desired temperature of 400 degrees.  After warming the oil it takes 5 minutes less time to cook than it took to warm up the oil.  How much time passes from starting the oil to having cooked chicken?'),
 (53,
  'James loves to go swimming and has to swim across a 20-mile lake.  He can swim at a pace of 2 miles per hour.  He swims 60% of the distance.  After that, he stops on an island and rests for half as long as the swimming time.  He then finishes the remaining distance while going half the speed.  How long did it take him to get across the lake?'),
 (84,
  'Every day Charisma meditates for 15 minutes when she first wakes up and again before she goes to sleep.  5 days a week she practices 1 hour of yoga.  in 4 weeks, how much time has she spent on meditation/yoga practice?')]